In [12]:
import cv2, dlib, librosa, numpy as np
import pandas as pd
from xgboost import XGBClassifier
import joblib
import os

# 🔧 Load models
print("🧠 Loading models...")
face_model = XGBClassifier()
face_model.load_model("xgb_face_model.json")
stack_model = joblib.load("stacked_model.pkl")

# 📍 Load dlib detectors
print("📍 Loading dlib face detector...")
face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# 🎞️ Input paths (EDIT THESE)
video_path = "../CroppedClips/clip_20250723_140404.avi"
audio_path = "../CroppedAudio/audio_20250723_140404.wav"
clip_name = os.path.basename(video_path)

if not os.path.exists(video_path):
    raise FileNotFoundError(f"❌ Video file not found: {video_path}")
if not os.path.exists(audio_path):
    raise FileNotFoundError(f"❌ Audio file not found: {audio_path}")

# 🎥 Extract facial landmarks every 30 frames
print(f"🎬 Processing video: {video_path}")
cap = cv2.VideoCapture(video_path)
landmarks = []
frame_count = 0
frames_seen = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frames_seen += 1
    if frame_count % 30 == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_detector(gray)
        if not faces:
            print(f"⚠️ No face detected at frame {frame_count}")
        else:
            shape = landmark_predictor(gray, faces[0])
            vec = np.array([[pt.x, pt.y] for pt in shape.parts()]).flatten()
            landmarks.append(vec)
    frame_count += 1
cap.release()

print(f"✅ Video processed: {frames_seen} frames scanned, {len(landmarks)} landmark frames extracted")

# 🧠 Facial score prediction
if not landmarks:
    print(f"🚫 No usable facial frames — skipping prediction for: {clip_name}")
    face_score = np.nan
else:
    avg_landmarks = np.mean(landmarks, axis=0)
    print(f"📏 Landmark vector shape: {avg_landmarks.shape}")
    if avg_landmarks.shape[0] != face_model.get_booster().num_features():
        raise ValueError(f"❌ Landmark shape mismatch. Expected {face_model.get_booster().num_features()}, got {avg_landmarks.shape[0]}")
    face_score = face_model.predict_proba(avg_landmarks.reshape(1, -1))[0][1]
    print(f"✅ Facial truth probability: {face_score:.4f}")

# 🔊 MFCC extraction
print(f"🔊 Processing audio: {audio_path}")
y, sr = librosa.load(audio_path, sr=None)
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
mfcc_vec = np.mean(mfcc, axis=1)
print(f"✅ MFCCs extracted: {mfcc_vec.shape} → {[round(v, 2) for v in mfcc_vec]}")

# 🧱 Build fusion input
fusion_input = pd.DataFrame({
    "face_score": [face_score],
    **{f"mfcc_{i}": [val] for i, val in enumerate(mfcc_vec)}
})

print("🔗 Fusion input shape:", fusion_input.shape)
print("📊 Fusion input preview:\n", fusion_input)

# 🚀 Run ensemble prediction
pred = stack_model.predict(fusion_input)[0]
conf = stack_model.predict_proba(fusion_input)[0][1]
label = "Truthful" if pred == 1 else "Deceptive"

print(f"\n🎞️ Final Result for {clip_name}: {label.upper()} (Confidence: {conf:.2f})")


🧠 Loading models...
📍 Loading dlib face detector...
🎬 Processing video: ../CroppedClips/clip_20250723_140404.avi
✅ Video processed: 62 frames scanned, 3 landmark frames extracted
📏 Landmark vector shape: (136,)
✅ Facial truth probability: 0.0154
🔊 Processing audio: ../CroppedAudio/audio_20250723_140404.wav
✅ MFCCs extracted: (13,) → [-410.66, 31.02, 5.33, 5.71, -6.51, -0.48, -15.27, -15.74, -10.07, -10.41, -5.59, 9.18, -7.12]
🔗 Fusion input shape: (1, 14)
📊 Fusion input preview:
    face_score      mfcc_0     mfcc_1    mfcc_2   mfcc_3    mfcc_4    mfcc_5  \
0     0.01536 -410.659576  31.020552  5.334141  5.70877 -6.509646 -0.479817   

      mfcc_6     mfcc_7     mfcc_8     mfcc_9   mfcc_10  mfcc_11   mfcc_12  
0 -15.269638 -15.740704 -10.071469 -10.409552 -5.589618  9.17749 -7.115943  

🎞️ Final Result for clip_20250723_140404.avi: DECEPTIVE (Confidence: 0.01)
